In [ ]:
include("../functions/functions_SCising_model_ALLsweeps_OG.jl")

using JLD2
using LinearAlgebra   # Symmetric, for the symmetric GRNs

base_path = "/home/twwoodw/my_folder/compute/data/results1D"
#base_path = "/data/scratch/projects/punim0638/woodward/nonrcep_casestudy"

function make_filenameSC(base_path, varname, K, β, σ, max_sweeps)
    return joinpath(base_path,
        "$(varname)1D_$(K)_$(β)_$(σ)_ALL$(max_sweeps)sweeps.jld2")
end

In [ ]:
# GRNs: same matrices as the multicellular study (run_ising_Wbank_V3.jl)
K_tree = Float64.([
    0  1  1  0  0  0  0  0  0;
    0  0  0  1  0  0  0  0  0;
    0  0  0  0  1  1  0  0  0;
    0  0  0  0  0  0 -1  0  0;
    0  0  0  0  0  0  0 -1  0;
    0  0  0  0  0  0  0  0 -1;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0
])

K_negtri = Float64.([
    0   1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_ICFF = Float64.([
    0   1  -1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_CFF = Float64.([
    0   1   1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_negtritri = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  -1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_negtritwin = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

# symmetric versions
K_treesym       = Matrix(Symmetric(copy(K_tree), :U))       + Matrix(Symmetric(copy(K_tree), :L))
K_negtrisym     = Matrix(Symmetric(copy(K_negtri), :U))     + Matrix(Symmetric(copy(K_negtri), :L))
K_ICFFsym       = Matrix(Symmetric(copy(K_ICFF), :U))       + Matrix(Symmetric(copy(K_ICFF), :L))
K_CFFsym        = Matrix(Symmetric(copy(K_CFF), :U))        + Matrix(Symmetric(copy(K_CFF), :L))
K_negtritrisym  = Matrix(Symmetric(copy(K_negtritri), :U))  + Matrix(Symmetric(copy(K_negtritri), :L))
K_negtritwinsym = Matrix(Symmetric(copy(K_negtritwin), :U)) + Matrix(Symmetric(copy(K_negtritwin), :L));

grns = ["tree", "negtri", "ICFF", "CFF", "negtritwin", "negtritri"]

K_asym = Dict("tree" => K_tree,    "negtri" => K_negtri,    "ICFF" => K_ICFF,
              "CFF"  => K_CFF,     "negtritwin" => K_negtritwin,    "negtritri" => K_negtritri)
K_sym  = Dict("tree" => K_treesym, "negtri" => K_negtrisym, "ICFF" => K_ICFFsym,
              "CFF"  => K_CFFsym,  "negtritwin" => K_negtritwinsym, "negtritri" => K_negtritrisym)

In [ ]:
### ----------- constants ----------- ###
n_cells = 400
n_genes = 9 
n_sims = 20 
max_sweeps = 10_000
#save_every = 10 # save spin_lattice[1:save_every:max_sweeps] 


### spin lattice ###
ξ1 = [1,1,1,-1,-1,-1,-1,-1,-1]
ξ2 = [-1,-1,-1,1,1,1,-1,-1,-1]
ξ3 = [-1,-1,-1,-1,-1,-1,1,1,1]
ξ  = hcat(ξ1, ξ2, ξ3)

σ_neg1 = -1*ones(n_cells, n_genes) # all spin down
σ_pos1 = 1*ones(n_cells, n_genes)  # all spin down
#σ_AFM = antiferromagnetic_spin_matrix(n_cells, n_genes)
σ_rand= rand([-1, 1], n_cells, n_genes)

half_ξ1 = repeat(reshape(ξ1, 1, n_genes), Int(n_cells / 2), 1)
half_ξ2 = repeat(reshape(ξ2, 1, n_genes), Int(n_cells / 2), 1)
half_ξ3 = repeat(reshape(ξ3, 1, n_genes), Int(n_cells / 2), 1)

σ_ξ1ξ2 = vcat(half_ξ1, half_ξ2);
σ_ξ2ξ1 = vcat(half_ξ2, half_ξ1);

In [ ]:
β_test = Inf
β_name = "βInf"

# true: every sim starts from its own random tissue (drawn fresh for each sim and each GRN)
# false: all sims start from the same tissue σ_test
random_init = true

σ_test = σ_rand
σ_name = random_init ? "σrandsim" : "σrand"   # new name, so the old shared-start results are not overwritten

In [ ]:
# run every GRN, asymmetric and symmetric; results are saved under K_name = name or "$(name)sym"
# GRNs whose result files already exist are skipped, except those listed in `rerun`
# (e.g. rerun = Set(["ICFF", "CFFsym"]) to redo those).
rerun = Set{String}()

max_sweeps_name = "$(max_sweeps)"
save_names = ["E_histories_selfish", "sigma_histories_selfish", "sigma_finals_selfish"]

for sym in (false, true), name in grns
    K_name = sym ? "$(name)sym" : name
    K_test = sym ? K_sym[name] : K_asym[name]

    files = Dict(v => make_filenameSC(base_path, v, K_name, β_name, σ_name, max_sweeps) for v in save_names)
    if all(isfile, values(files)) && !(K_name in rerun)
        println("$(K_name): results exist, skipped")
        continue
    end

    println("running $(K_name)")
    # one run_nsims call per sim, each from its own starting tissue
    @time results = [
        run_nsims(
            random_init ? rand([-1, 1], n_cells, n_genes) : copy(σ_test);
            K_gene = K_test,
            β = β_test,
            n_sims = 1,
            max_sweeps = max_sweeps
        )
        for _ in 1:n_sims
    ]

    vars = Dict(
        "E_histories_selfish"     => reduce(vcat, [r.E_histories_selfish for r in results]),
        "sigma_histories_selfish" => reduce(vcat, [r.σ_histories_selfish for r in results]),
        "sigma_finals_selfish"    => reduce(vcat, [r.σ_finals_selfish    for r in results])
    )

    for (var, val) in vars
        jldsave(files[var]; val)
    end
end